In [1]:
import pandas as pd
import numpy as np

matches_df = pd.read_csv('../data/processed/matches.csv')

league_trends = matches_df.groupby(['league', 'year']).agg(
    matches=('match_id', 'count'),
    avg_goals=('total_goals', 'mean'),
    avg_xG=('total_xG', 'mean')
).round(3)

print(league_trends)

                     matches  avg_goals  avg_xG
league         year                            
bundesliga     2020      306      3.033   2.860
               2021      306      3.118   3.120
               2022      306      3.173   3.027
               2023      306      3.219   3.336
               2024      306      3.134   3.261
la_liga        2020      380      2.508   2.504
               2021      380      2.503   2.631
               2022      380      2.513   2.774
               2023      380      2.645   2.875
               2024      380      2.618   2.850
ligue_1        2020      380      2.761   2.658
               2021      380      2.808   2.718
               2022      380      2.808   2.944
               2023      306      2.699   2.964
               2024      306      2.977   3.297
premier_league 2020      380      2.695   2.735
               2021      380      2.818   2.839
               2022      380      2.853   2.965
               2023      380      3.279 

In [2]:
shots_df = pd.read_csv("/Users/aadvikmazumdar/Projects/OnTarget/data/processed/shots_enriched.csv")
shots_by_league_season = shots_df.groupby(['league', 'year']).size().reset_index(name='total_shots')

league_full = matches_df.groupby(['league', 'year']).agg(
    matches=('match_id', 'count'),
    total_goals=('total_goals', 'sum'),
    total_xG=('total_xG', 'sum')
).reset_index()

league_full = league_full.merge(shots_by_league_season, on=['league', 'year'])

league_full['avg_shots_per_match'] = (league_full['total_shots'] / league_full['matches']).round(2)
league_full['avg_goals_per_match'] = (league_full['total_goals'] / league_full['matches']).round(3)
league_full['avg_xG_per_match'] = (league_full['total_xG'] / league_full['matches']).round(3)
league_full['shot_conversion'] = (league_full['total_goals'] / league_full['total_shots']).round(4)

print(league_full[['league', 'year', 'avg_shots_per_match', 'avg_goals_per_match', 'avg_xG_per_match', 'shot_conversion']])

            league  year  avg_shots_per_match  avg_goals_per_match  \
0       bundesliga  2020                24.80                3.033   
1       bundesliga  2021                25.98                3.118   
2       bundesliga  2022                25.54                3.173   
3       bundesliga  2023                27.80                3.219   
4       bundesliga  2024                25.96                3.134   
5          la_liga  2020                21.40                2.508   
6          la_liga  2021                23.62                2.503   
7          la_liga  2022                24.67                2.513   
8          la_liga  2023                24.50                2.645   
9          la_liga  2024                23.83                2.618   
10         ligue_1  2020                23.48                2.761   
11         ligue_1  2021                24.03                2.808   
12         ligue_1  2022                24.59                2.808   
13         ligue_1  

In [3]:
pd.set_option('display.max_rows', None)
print(league_full[['league', 'year', 'avg_shots_per_match', 'avg_goals_per_match', 'avg_xG_per_match', 'shot_conversion']])

            league  year  avg_shots_per_match  avg_goals_per_match  \
0       bundesliga  2020                24.80                3.033   
1       bundesliga  2021                25.98                3.118   
2       bundesliga  2022                25.54                3.173   
3       bundesliga  2023                27.80                3.219   
4       bundesliga  2024                25.96                3.134   
5          la_liga  2020                21.40                2.508   
6          la_liga  2021                23.62                2.503   
7          la_liga  2022                24.67                2.513   
8          la_liga  2023                24.50                2.645   
9          la_liga  2024                23.83                2.618   
10         ligue_1  2020                23.48                2.761   
11         ligue_1  2021                24.03                2.808   
12         ligue_1  2022                24.59                2.808   
13         ligue_1  

In [4]:
def flag_season_anomalies(df, metric):
    stats = df.groupby('league')[metric].agg(['mean', 'std']).rename(columns={'mean': f'{metric}_mean', 'std': f'{metric}_std'})
    merged = df.merge(stats, on='league')
    merged[f'{metric}_zscore'] = (merged[metric] - merged[f'{metric}_mean']) / merged[f'{metric}_std']
    return merged[['league', 'year', metric, f'{metric}_zscore']]

shots_anomalies = flag_season_anomalies(league_full, 'avg_shots_per_match')
print(shots_anomalies[shots_anomalies['avg_shots_per_match_zscore'].abs() > 1.0].sort_values('avg_shots_per_match_zscore', ascending=False))

            league  year  avg_shots_per_match  avg_shots_per_match_zscore
3       bundesliga  2023                27.80                    1.613123
18  premier_league  2023                27.69                    1.525075
13         ligue_1  2023                25.58                    1.280254
21         serie_a  2021                26.14                    1.218016
0       bundesliga  2020                24.80                   -1.099528
15  premier_league  2020                24.30                   -1.233732
10         ligue_1  2020                23.48                   -1.290046
24         serie_a  2024                24.32                   -1.459265
5          la_liga  2020                21.40                   -1.684441


In [5]:
print(shots_df[shots_df['year'] == 2023].groupby('league')['match_id'].nunique())
print()
print(matches_df[matches_df['year'] == 2023].groupby('league')['match_id'].nunique())

league
bundesliga        306
la_liga           380
ligue_1           306
premier_league    380
serie_a           380
Name: match_id, dtype: int64

league
bundesliga        306
la_liga           380
ligue_1           306
premier_league    380
serie_a           380
Name: match_id, dtype: int64


In [6]:
team_season_shots = shots_df.groupby(['team_name', 'league', 'year']).agg(
    total_shots=('team_name', 'count'),
    total_goals=('result', lambda x: (x == 'Goal').sum()),
    avg_xG=('xG', 'mean')
).reset_index()

team_season_shots['conversion'] = (team_season_shots['total_goals'] / team_season_shots['total_shots']).round(4)

# z-score within each league-season (not across the whole dataset)
def zscore_within_group(df, metric, group_cols):
    stats = df.groupby(group_cols)[metric].agg(['mean', 'std'])
    merged = df.merge(stats, on=group_cols)
    merged[f'{metric}_zscore'] = (merged[metric] - merged['mean']) / merged['std']
    return merged.drop(columns=['mean', 'std'])

team_season_shots = zscore_within_group(team_season_shots, 'total_shots', ['league', 'year'])
team_season_shots = zscore_within_group(team_season_shots, 'conversion', ['league', 'year'])

print(team_season_shots.sort_values('total_shots_zscore', ascending=False).head(10))
print()
print(team_season_shots.sort_values('conversion_zscore', ascending=False).head(10))

               team_name          league  year  total_shots  total_goals  \
64         Bayern Munich      bundesliga  2024          646           96   
355          Real Madrid         la_liga  2021          657           80   
61         Bayern Munich      bundesliga  2021          673           92   
62         Bayern Munich      bundesliga  2022          630           90   
337  Paris Saint Germain         ligue_1  2024          638           90   
251            Liverpool  premier_league  2023          792           80   
258                 Lyon         ligue_1  2020          614           77   
311               Napoli         serie_a  2023          650           55   
54             Barcelona         la_liga  2024          677           99   
249            Liverpool  premier_league  2021          730           94   

       avg_xG  conversion  total_shots_zscore  conversion_zscore  
64   0.148132      0.1486            3.214533           1.372746  
355  0.124496      0.1218    

In [7]:
team_season_shots['combined_zscore'] = team_season_shots['total_shots_zscore'] + team_season_shots['conversion_zscore']
print(team_season_shots.sort_values('combined_zscore', ascending=False).head(10))

               team_name      league  year  total_shots  total_goals  \
64         Bayern Munich  bundesliga  2024          646           96   
54             Barcelona     la_liga  2024          677           99   
60         Bayern Munich  bundesliga  2020          576           98   
355          Real Madrid     la_liga  2021          657           80   
334  Paris Saint Germain     ligue_1  2021          564           88   
62         Bayern Munich  bundesliga  2022          630           90   
336  Paris Saint Germain     ligue_1  2023          513           78   
61         Bayern Munich  bundesliga  2021          673           92   
337  Paris Saint Germain     ligue_1  2024          638           90   
333  Paris Saint Germain     ligue_1  2020          569           85   

       avg_xG  conversion  total_shots_zscore  conversion_zscore  \
64   0.148132      0.1486            3.214533           1.372746   
54   0.148488      0.1462            2.484813           1.941551   
60 

In [8]:
anomaly_counts = team_season_shots[
    (team_season_shots['total_shots_zscore'] > 1.5) | (team_season_shots['conversion_zscore'] > 1.5)
].groupby('team_name').size().sort_values(ascending=False)

print(anomaly_counts.head(15))

team_name
Real Madrid               5
Paris Saint Germain       5
Bayern Munich             5
Inter                     5
Manchester City           4
Liverpool                 4
Napoli                    3
Barcelona                 3
Monaco                    2
Lyon                      2
RasenBallsport Leipzig    2
Lazio                     2
Fiorentina                2
Borussia Dortmund         2
Bayer Leverkusen          2
dtype: int64


In [9]:
efficient_low_volume = team_season_shots[
    (team_season_shots['total_shots_zscore'] < 0) & (team_season_shots['conversion_zscore'] > 1.0)
].sort_values('conversion_zscore', ascending=False)

print(efficient_low_volume)

                  team_name          league  year  total_shots  total_goals  \
219                   Lazio         serie_a  2021          455           74   
93                Brentford  premier_league  2024          444           64   
220                   Lazio         serie_a  2022          439           59   
115              Celta Vigo         la_liga  2020          359           55   
423               Tottenham  premier_league  2020          447           66   
341  RasenBallsport Leipzig      bundesliga  2021          438           72   
233               Leicester  premier_league  2021          435           62   
169              Fiorentina         serie_a  2024          448           58   
417              Strasbourg         ligue_1  2024          359           54   
297             Montpellier         ligue_1  2022          445           64   
380                    Roma         serie_a  2023          479           64   
451                  Verona         serie_a  2021   

In [10]:
team_total_goals = shots_df.groupby('team_name').agg(
    total_goals=('result', lambda x: (x == 'Goal').sum()),
    total_shots=('team_name', 'count')
).sort_values('total_goals', ascending=False)

print(team_total_goals.head(15))

                     total_goals  total_shots
team_name                                    
Bayern Munich                469         3163
Manchester City              435         3222
Paris Saint Germain          427         2854
Inter                        398         3031
Liverpool                    395         3388
Barcelona                    392         2947
Real Madrid                  379         3086
Borussia Dortmund            373         2508
Atalanta                     364         2837
Arsenal                      349         2845
Napoli                       343         2996
Bayer Leverkusen             341         2473
Monaco                       337         2447
AC Milan                     334         2847
Atletico Madrid              331         2418


In [11]:
h2h_counts = matches_df.groupby(['home_team', 'away_team']).size().reset_index(name='meetings')

# combine home+away perspective into one fixture pair (order-independent)
h2h_counts['fixture_pair'] = h2h_counts.apply(
    lambda r: tuple(sorted([r['home_team'], r['away_team']])), axis=1
)

h2h_summary = h2h_counts.groupby('fixture_pair')['meetings'].sum().sort_values(ascending=False)
print(h2h_summary.head(15))

fixture_pair
(AC Milan, Atalanta)                         10
(Inter, Lazio)                               10
(Chelsea, Tottenham)                         10
(Chelsea, West Ham)                          10
(Chelsea, Wolverhampton Wanderers)           10
(Crystal Palace, Everton)                    10
(Crystal Palace, Liverpool)                  10
(Crystal Palace, Manchester City)            10
(Crystal Palace, Manchester United)          10
(Crystal Palace, Newcastle United)           10
(Crystal Palace, Tottenham)                  10
(Crystal Palace, West Ham)                   10
(Crystal Palace, Wolverhampton Wanderers)    10
(Eintracht Frankfurt, Freiburg)              10
(Eintracht Frankfurt, Hoffenheim)            10
Name: meetings, dtype: int64


In [12]:
# build head-to-head goals: for each match, tag both teams' goals against each other
h2h_matches = matches_df[['home_team', 'away_team', 'home_goals', 'away_goals', 'league', 'year']].copy()

h2h_matches['fixture_pair'] = h2h_matches.apply(
    lambda r: tuple(sorted([r['home_team'], r['away_team']])), axis=1
)

# unpivot into one row per team per match (so we can sum goals per team per pairing)
home_rows = h2h_matches.rename(columns={'home_team': 'team_name', 'away_team': 'opponent', 'home_goals': 'goals_scored'})[['team_name', 'opponent', 'goals_scored', 'fixture_pair']]
away_rows = h2h_matches.rename(columns={'away_team': 'team_name', 'home_team': 'opponent', 'away_goals': 'goals_scored'})[['team_name', 'opponent', 'goals_scored', 'fixture_pair']]

h2h_long = pd.concat([home_rows, away_rows], ignore_index=True)

h2h_goals = h2h_long.groupby(['team_name', 'opponent', 'fixture_pair']).agg(
    goals_scored=('goals_scored', 'sum'),
    meetings=('goals_scored', 'count')
).reset_index()

h2h_goals['goals_per_meeting'] = (h2h_goals['goals_scored'] / h2h_goals['meetings']).round(2)

print(h2h_goals.sort_values('goals_scored', ascending=False).head(15))

                team_name                 opponent  \
2169  Paris Saint Germain              Montpellier   
365         Bayern Munich                   Bochum   
485     Borussia Dortmund                 Freiburg   
377         Bayern Munich                 Mainz 05   
1668            Liverpool                Tottenham   
382         Bayern Munich            VfB Stuttgart   
1805      Manchester City         Newcastle United   
2163  Paris Saint Germain                    Lille   
1855            Marseille              Montpellier   
1814      Manchester City  Wolverhampton Wanderers   
331             Barcelona               Real Betis   
337             Barcelona                 Valencia   
2176  Paris Saint Germain               Strasbourg   
480     Borussia Dortmund      Borussia M.Gladbach   
1662            Liverpool        Manchester United   

                                    fixture_pair  goals_scored  meetings  \
2169          (Montpellier, Paris Saint Germain)           

In [13]:
players_df = pd.read_csv('../data/processed/players.csv')
print(players_df.columns.tolist())


['id', 'player_name', 'games', 'time', 'goals', 'xG', 'assists', 'xA', 'shots', 'key_passes', 'yellow_cards', 'red_cards', 'position', 'team_title', 'npg', 'npxG', 'xGChain', 'xGBuildup', 'league', 'year']


In [14]:
players_df['primary_position'] = players_df['position'].str.split().str[0]
print(players_df['primary_position'].value_counts())

primary_position
D     5166
F     3155
M     2865
S     1795
GK     983
Name: count, dtype: int64


In [15]:
print(players_df.columns.tolist())

['id', 'player_name', 'games', 'time', 'goals', 'xG', 'assists', 'xA', 'shots', 'key_passes', 'yellow_cards', 'red_cards', 'position', 'team_title', 'npg', 'npxG', 'xGChain', 'xGBuildup', 'league', 'year', 'primary_position']


In [16]:
players_df['per90_reliable'] = players_df['time'] >= 450
players_df['npxG_per90'] = (players_df['npxG'] / players_df['time'] * 90).round(3)
players_df['goals_per90'] = (players_df['goals'] / players_df['time'] * 90).round(3)
players_df['xGChain_per90'] = (players_df['xGChain'] / players_df['time'] * 90).round(3)
players_df['xA_per90'] = (players_df['xA'] / players_df['time'] * 90).round(3)
players_df['key_passes_per90'] = (players_df['key_passes'] / players_df['time'] * 90).round(3)

print(players_df.columns.tolist())

['id', 'player_name', 'games', 'time', 'goals', 'xG', 'assists', 'xA', 'shots', 'key_passes', 'yellow_cards', 'red_cards', 'position', 'team_title', 'npg', 'npxG', 'xGChain', 'xGBuildup', 'league', 'year', 'primary_position', 'per90_reliable', 'npxG_per90', 'goals_per90', 'xGChain_per90', 'xA_per90', 'key_passes_per90']


In [17]:
anomaly_pool = players_df[(players_df['per90_reliable']) & (players_df['primary_position'] != 'S')].copy()

def zscore_within_group(df, metric, group_cols):
    stats = df.groupby(group_cols)[metric].agg(['mean', 'std'])
    merged = df.merge(stats, on=group_cols, suffixes=('', '_stat'))
    merged[f'{metric}_zscore'] = (merged[metric] - merged['mean']) / merged['std']
    return merged.drop(columns=['mean', 'std'])

metrics = ['npxG_per90', 'goals_per90', 'xGChain_per90', 'xA_per90', 'key_passes_per90']
for m in metrics:
    anomaly_pool = zscore_within_group(anomaly_pool, m, ['primary_position', 'league', 'year'])

print(anomaly_pool[anomaly_pool['npxG_per90_zscore'] > 2].sort_values('npxG_per90_zscore', ascending=False)[['player_name', 'team_title', 'league', 'year', 'primary_position', 'npxG_per90', 'npxG_per90_zscore']].head(15))

              player_name         team_title          league  year  \
4699    Zakaria Aboukhlal           Toulouse         ligue_1  2022   
7143   Keane Lewis-Potter          Brentford  premier_league  2023   
9118      Davide Frattesi              Inter         serie_a  2023   
6251    Christian Pulisic            Chelsea  premier_league  2021   
7460         Noni Madueke            Chelsea  premier_league  2024   
6159  Alireza Jahanbakhsh           Brighton  premier_league  2020   
1496        Donyell Malen  Borussia Dortmund      bundesliga  2024   
1865    Philippe Coutinho          Barcelona         la_liga  2020   
6719   Alejandro Garnacho  Manchester United  premier_league  2022   
3541          Samuel Lino    Atletico Madrid         la_liga  2024   
5132         Yusuf Yazici              Lille         ligue_1  2023   
361          Serge Gnabry      Bayern Munich      bundesliga  2021   
3037     Cristhian Stuani             Girona         la_liga  2023   
3806          Joan G

In [18]:
check_players = ['Zakaria Aboukhlal', 'Christian Pulisic', 'Noni Madueke', 'Serge Gnabry']
raw_check = players_df[players_df['player_name'].isin(check_players)]
print(raw_check[['player_name', 'position', 'primary_position']].to_string())

             player_name position primary_position
11          Serge Gnabry      M S                M
502         Serge Gnabry    D M S                D
1011        Serge Gnabry    F M S                F
1609        Serge Gnabry      M S                M
2042        Serge Gnabry    F M S                F
6618   Zakaria Aboukhlal  D F M S                D
7278   Zakaria Aboukhlal    F M S                F
7736   Zakaria Aboukhlal    F M S                F
8322   Christian Pulisic    F M S                F
8820   Christian Pulisic  D F M S                D
9514   Christian Pulisic  D F M S                D
9577        Noni Madueke    F M S                F
9947        Noni Madueke      M S                M
10484       Noni Madueke    D M S                D
12785  Christian Pulisic    F M S                F
13375  Christian Pulisic    F M S                F


In [19]:
print(players_df[players_df['player_name'] == 'Serge Gnabry'][['player_name', 'league', 'year', 'team_title', 'time', 'position']])

       player_name      league  year     team_title  time position
11    Serge Gnabry  bundesliga  2020  Bayern Munich  1653      M S
502   Serge Gnabry  bundesliga  2021  Bayern Munich  2205    D M S
1011  Serge Gnabry  bundesliga  2022  Bayern Munich  1950    F M S
1609  Serge Gnabry  bundesliga  2023  Bayern Munich   431      M S
2042  Serge Gnabry  bundesliga  2024  Bayern Munich  1216    F M S


In [20]:
def resolve_position(pos_string):
    tokens = set(pos_string.split())
    if 'GK' in tokens:
        return 'GK'
    elif 'F' in tokens:
        return 'F'
    elif 'M' in tokens:
        return 'M'
    elif 'D' in tokens:
        return 'D'
    else:
        return 'S'

players_df['primary_position_hierarchy'] = players_df['position'].apply(resolve_position)
print(players_df['primary_position_hierarchy'].value_counts())

print(players_df[players_df['player_name'] == 'Serge Gnabry'][['player_name', 'year', 'position', 'primary_position_hierarchy']])

primary_position_hierarchy
M     4174
D     3548
F     3464
S     1795
GK     983
Name: count, dtype: int64
       player_name  year position primary_position_hierarchy
11    Serge Gnabry  2020      M S                          M
502   Serge Gnabry  2021    D M S                          M
1011  Serge Gnabry  2022    F M S                          F
1609  Serge Gnabry  2023      M S                          M
2042  Serge Gnabry  2024    F M S                          F


In [22]:
# aggregate actual minutes played per position, per player-season, from rosters_df
rosters_df = pd.read_csv("/Users/aadvikmazumdar/Projects/OnTarget/data/processed/rosters.csv")
minutes_by_position = rosters_df.groupby(['player_id', 'league', 'year', 'position'])['time'].sum().reset_index()

# map rosters_df's detailed codes (DC, DL, DML etc.) to the same broad buckets
def broad_position(code):
    if code == 'GK':
        return 'GK'
    elif code.startswith('D'):
        return 'D'
    elif code.startswith('AM') or code in ['MC', 'ML', 'MR']:
        return 'M'
    elif code.startswith('FW'):
        return 'F'
    else:
        return 'Other'

minutes_by_position['broad_position'] = minutes_by_position['position'].apply(broad_position)

# sum minutes by broad position, then pick whichever bucket has the MOST actual minutes
minutes_summed = minutes_by_position.groupby(['player_id', 'league', 'year', 'broad_position'])['time'].sum().reset_index()

primary_by_minutes = minutes_summed.loc[
    minutes_summed.groupby(['player_id', 'league', 'year'])['time'].idxmax()
][['player_id', 'league', 'year', 'broad_position']].rename(columns={'broad_position': 'primary_position_minutes'})

print(primary_by_minutes.head(10))

    player_id      league  year primary_position_minutes
0           3     serie_a  2020                        D
2           3     serie_a  2022                        D
4           3     serie_a  2023                        D
6           3     serie_a  2024                        D
9           5  bundesliga  2023                        M
12          9  bundesliga  2023                        M
14         19  bundesliga  2020                       GK
15         22  bundesliga  2020                        D
17         22  bundesliga  2021                        D
19         22  bundesliga  2022                        D


In [23]:
# merge players_df id column with rosters_df's player_id — confirm they're the same key first
print(players_df[['id', 'player_name']].head())
print(rosters_df[['player_id', 'player']].head())

     id         player_name
0   227  Robert Lewandowski
1  6170         André Silva
2  8260      Erling Haaland
3   956     Andrej Kramaric
4  7052       Wout Weghorst
   player_id          player
0       8715   Illan Meslier
1       8716     Luke Ayling
2       8816     Liam Cooper
3       6273      Robin Koch
4       8722  Ezgjan Alioski


In [24]:
players_ids = set(players_df['id'].unique())
rosters_ids = set(rosters_df['player_id'].unique())

overlap = players_ids & rosters_ids
print(f"players_df unique ids: {len(players_ids)}")
print(f"rosters_df unique ids: {len(rosters_ids)}")
print(f"overlapping ids: {len(overlap)}")

players_df unique ids: 5630
rosters_df unique ids: 5630
overlapping ids: 5630


In [25]:
comparison = players_df[['id', 'player_name', 'league', 'year', 'primary_position_hierarchy']].merge(
    primary_by_minutes.rename(columns={'player_id': 'id'}),
    on=['id', 'league', 'year'],
    how='inner'
)

comparison['agree'] = comparison['primary_position_hierarchy'] == comparison['primary_position_minutes']

print(comparison['agree'].value_counts())
print()
print(comparison['agree'].value_counts(normalize=True).round(3))

agree
True     8720
False    5244
Name: count, dtype: int64

agree
True     0.624
False    0.376
Name: proportion, dtype: float64


In [26]:
players_df = players_df.merge(
    primary_by_minutes.rename(columns={'player_id': 'id'}),
    on=['id', 'league', 'year'],
    how='left'
)

print(players_df['primary_position_minutes'].isna().sum(), 'unmatched players')

anomaly_pool = players_df[(players_df['per90_reliable']) & (players_df['primary_position_minutes'].notna()) & (players_df['primary_position_minutes'] != 'Other')].copy()

def zscore_within_group(df, metric, group_cols):
    stats = df.groupby(group_cols)[metric].agg(['mean', 'std'])
    merged = df.merge(stats, on=group_cols, suffixes=('', '_stat'))
    merged[f'{metric}_zscore'] = (merged[metric] - merged['mean']) / merged['std']
    return merged.drop(columns=['mean', 'std'])

metrics = ['npxG_per90', 'goals_per90', 'xGChain_per90', 'xA_per90', 'key_passes_per90']
for m in metrics:
    anomaly_pool = zscore_within_group(anomaly_pool, m, ['primary_position_minutes', 'league', 'year'])

print(anomaly_pool[anomaly_pool['npxG_per90_zscore'] > 2].sort_values('npxG_per90_zscore', ascending=False)[['player_name', 'team_title', 'league', 'year', 'primary_position_minutes', 'npxG_per90', 'npxG_per90_zscore']].head(15))

0 unmatched players
            player_name           team_title          league  year  \
7389       Ross Barkley          Aston Villa  premier_league  2024   
8961    Davide Frattesi                Inter         serie_a  2023   
1828  Philippe Coutinho            Barcelona         la_liga  2020   
2652  Ander Barrenetxea        Real Sociedad         la_liga  2022   
6251       Matt Doherty            Tottenham  premier_league  2021   
1056   Jeremie Frimpong     Bayer Leverkusen      bundesliga  2023   
6563        Deniz Undav             Brighton  premier_league  2022   
1092       Nathan Tella     Bayer Leverkusen      bundesliga  2023   
4306       Sergio Ramos  Paris Saint Germain         ligue_1  2021   
404       Jamal Musiala        Bayern Munich      bundesliga  2021   
3735        Joan García             Espanyol         la_liga  2024   
7575               Kepa          Bournemouth  premier_league  2024   
2924   Jeremías Ledesma                Cadiz         la_liga  2022   


In [27]:
anomaly_pool = players_df[
    (players_df['per90_reliable']) &
    (players_df['primary_position_minutes'].notna()) &
    (~players_df['primary_position_minutes'].isin(['Other', 'GK']))
].copy()

for m in metrics:
    anomaly_pool = zscore_within_group(anomaly_pool, m, ['primary_position_minutes', 'league', 'year'])

print(anomaly_pool[anomaly_pool['npxG_per90_zscore'] > 2].sort_values('npxG_per90_zscore', ascending=False)[['player_name', 'team_title', 'league', 'year', 'primary_position_minutes', 'npxG_per90', 'npxG_per90_zscore']].head(15))

            player_name           team_title          league  year  \
6860       Ross Barkley          Aston Villa  premier_league  2024   
8309    Davide Frattesi                Inter         serie_a  2023   
1702  Philippe Coutinho            Barcelona         la_liga  2020   
2467  Ander Barrenetxea        Real Sociedad         la_liga  2022   
5808       Matt Doherty            Tottenham  premier_league  2021   
983    Jeremie Frimpong     Bayer Leverkusen      bundesliga  2023   
6094        Deniz Undav             Brighton  premier_league  2022   
1019       Nathan Tella     Bayer Leverkusen      bundesliga  2023   
4000       Sergio Ramos  Paris Saint Germain         ligue_1  2021   
383       Jamal Musiala        Bayern Munich      bundesliga  2021   
3961       Pereira Lage               Angers         ligue_1  2021   
8507          Emil Holm             Atalanta         serie_a  2023   
3533             Neymar  Paris Saint Germain         ligue_1  2020   
7144       Robin Gos

In [28]:
print(anomaly_pool.groupby('primary_position_minutes')['npxG_per90'].agg(['mean', 'std', 'count']).round(4))

                            mean     std  count
primary_position_minutes                       
D                         0.0529  0.0463   4270
F                         0.3560  0.1653   1540
M                         0.1402  0.1082   3211


In [29]:
players_df.to_csv('../data/processed/players_enriched.csv', index=False)
print("saved players_enriched.csv with shape:", players_df.shape)

saved players_enriched.csv with shape: (13964, 29)


In [30]:
player_season_counts = players_df.groupby('id')['year'].nunique()
multi_season_players = player_season_counts[player_season_counts >= 3].index

peak_pool = players_df[
    (players_df['id'].isin(multi_season_players)) &
    (players_df['per90_reliable'])
].copy()

print(f"Players with 3+ reliable seasons: {peak_pool['id'].nunique()}")
print(peak_pool.shape)

Players with 3+ reliable seasons: 2215
(7739, 29)


In [31]:
def player_own_zscore(df, metric):
    stats = df.groupby('id')[metric].agg(['mean', 'std']).rename(columns={'mean': f'{metric}_own_mean', 'std': f'{metric}_own_std'})
    merged = df.merge(stats, on='id')
    merged[f'{metric}_own_zscore'] = (merged[metric] - merged[f'{metric}_own_mean']) / merged[f'{metric}_own_std']
    return merged

peak_pool = player_own_zscore(peak_pool, 'npxG_per90')

peak_pool['peak_season'] = peak_pool['npxG_per90_own_zscore'] > 1.5

print(peak_pool['peak_season'].sum(), 'peak seasons flagged')
print()
print(peak_pool[peak_pool['peak_season']].sort_values('npxG_per90_own_zscore', ascending=False)[
    ['player_name', 'team_title', 'league', 'year', 'npxG_per90', 'npxG_per90_own_mean', 'npxG_per90_own_zscore']
].head(15))

207 peak seasons flagged

                 player_name               team_title          league  year  \
3522               Matz Sels               Strasbourg         ligue_1  2021   
2465                Reinildo          Atletico Madrid         la_liga  2023   
4230           Marco Asensio      Paris Saint Germain         ligue_1  2024   
4757         Bruno Guimarães         Newcastle United  premier_league  2021   
1564   Marc-André ter Stegen                Barcelona         la_liga  2020   
5674               Nick Pope         Newcastle United  premier_league  2023   
1529         Marko Dmitrovic                    Eibar         la_liga  2020   
6657  Vanja Milinkovic-Savic                   Torino         serie_a  2021   
1885               Jan Oblak          Atletico Madrid         la_liga  2021   
4534                 Alisson                Liverpool  premier_league  2020   
785             Manuel Neuer            Bayern Munich      bundesliga  2022   
6549           Martin Hong

In [32]:
peak_pool = players_df[
    (players_df['id'].isin(multi_season_players)) &
    (players_df['per90_reliable']) &
    (players_df['primary_position_minutes'] != 'GK')
].copy()

peak_pool = player_own_zscore(peak_pool, 'npxG_per90')
peak_pool['peak_season'] = peak_pool['npxG_per90_own_zscore'] > 1.5

print(peak_pool['peak_season'].sum(), 'peak seasons flagged')
print()
print(peak_pool[peak_pool['peak_season']].sort_values('npxG_per90_own_zscore', ascending=False)[
    ['player_name', 'team_title', 'league', 'year', 'npxG_per90', 'npxG_per90_own_mean', 'npxG_per90_own_zscore']
].head(15))

197 peak seasons flagged

           player_name               team_title          league  year  \
2295          Reinildo          Atletico Madrid         la_liga  2023   
3941     Marco Asensio      Paris Saint Germain         ligue_1  2024   
4429   Bruno Guimarães         Newcastle United  premier_league  2021   
6101     Martin Hongla                   Verona         serie_a  2021   
1005        Kevin Vogt  Hoffenheim,Union Berlin      bundesliga  2023   
3672   Youssouf Fofana                   Monaco         ligue_1  2023   
2041         Loic Bade                  Sevilla         la_liga  2022   
6045   Davide Calabria                 AC Milan         serie_a  2021   
4706      Granit Xhaka                  Arsenal  premier_league  2022   
5382      Mikel Merino                  Arsenal  premier_league  2024   
65            Angelino   RasenBallsport Leipzig      bundesliga  2020   
6250       Rafael Leão                 AC Milan         serie_a  2022   
3152  Ibrahima Sissoko   

In [33]:
print(peak_pool.groupby('primary_position_minutes')['npxG_per90'].agg(['mean', 'std', 'count']).round(4))

                            mean     std  count
primary_position_minutes                       
D                         0.0556  0.0479   3359
F                         0.3659  0.1677   1210
M                         0.1479  0.1116   2505
Other                     0.2545  0.1884    122


In [34]:
players_df['penalty_dependency'] = (players_df['xG'] - players_df['npxG']).round(3)

penalty_reliant = players_df[players_df['per90_reliable']].sort_values('penalty_dependency', ascending=False)
print(penalty_reliant[['player_name', 'team_title', 'league', 'year', 'goals', 'xG', 'npxG', 'penalty_dependency']].head(15))

                    player_name               team_title          league  \
11011             Franck Kessié                 AC Milan         serie_a   
11607           Lorenzo Insigne                   Napoli         serie_a   
5435          Wissam Ben Yedder                   Monaco         ligue_1   
6599   Jonathan Christian David                    Lille         ligue_1   
3058              Karim Benzema              Real Madrid         la_liga   
12780          Hakan Calhanoglu                    Inter         serie_a   
8251            Bruno Fernandes        Manchester United  premier_league   
2489              Gerard Moreno               Villarreal         la_liga   
11590             Ciro Immobile                    Lazio         serie_a   
10434             Mohamed Salah                Liverpool  premier_league   
8255                Jamie Vardy                Leicester  premier_league   
9865                Cole Palmer  Chelsea,Manchester City  premier_league   
8293        

In [35]:
players_df['penalty_dependency_pct'] = (players_df['penalty_dependency'] / players_df['xG'].replace(0, np.nan) * 100).round(1)

print(players_df[players_df['per90_reliable']].sort_values('penalty_dependency_pct', ascending=False)[
    ['player_name', 'team_title', 'league', 'year', 'xG', 'npxG', 'penalty_dependency_pct']
].head(15))

             player_name         team_title          league  year        xG  \
1699    Leonardo Bonucci       Union Berlin      bundesliga  2023  0.757777   
2177          Kevin Vogt       Union Berlin      bundesliga  2024  0.757777   
316       Nabil Bentaleb         Schalke 04      bundesliga  2020  0.757777   
944       Manuel Riemann             Bochum      bundesliga  2021  0.757777   
8293            Jorginho            Chelsea  premier_league  2020  6.972691   
2716     Marko Dmitrovic              Eibar         la_liga  2020  1.558895   
11644  Domenico Criscito              Genoa         serie_a  2021  5.659400   
540             Emre Can  Borussia Dortmund      bundesliga  2021  3.246970   
8817            Jorginho            Chelsea  premier_league  2021  5.913266   
5537          Kenny Lala         Strasbourg         ligue_1  2020  2.586738   
11226  Domenico Criscito              Genoa         serie_a  2020  0.868794   
4278              Pepelu           Valencia         

In [36]:
penalty_rate_filtered = players_df[(players_df['per90_reliable']) & (players_df['xG'] >= 3.0)].sort_values('penalty_dependency_pct', ascending=False)

print(penalty_rate_filtered[['player_name', 'team_title', 'league', 'year', 'xG', 'npxG', 'penalty_dependency_pct']].head(15))

             player_name         team_title          league  year         xG  \
8293            Jorginho            Chelsea  premier_league  2020   6.972691   
11644  Domenico Criscito              Genoa         serie_a  2021   5.659400   
540             Emre Can  Borussia Dortmund      bundesliga  2021   3.246970   
8817            Jorginho            Chelsea  premier_league  2021   5.913266   
4278              Pepelu           Valencia         la_liga  2023   6.946358   
11069      Nicolas Viola          Benevento         serie_a  2020   3.597570   
6049      Thomas Mangani             Angers         ligue_1  2021   5.437245   
6100     Florian Tardieu             Troyes         ligue_1  2021   3.686876   
1062   Maximilian Arnold          Wolfsburg      bundesliga  2022   4.634974   
1564       Florian Kainz         FC Cologne      bundesliga  2023   4.688287   
12780   Hakan Calhanoglu              Inter         serie_a  2023   9.977345   
12869    Leandro Paredes               R

In [37]:
print(shots_df[['player_id', 'match_id', 'h_a']].head())
print(rosters_df[['player_id', 'match_id', 'h_a', 'position']].head())

   player_id  match_id h_a
0       6018     13977   a
1       7169     13977   a
2       6018     13977   a
3       6902     13977   h
4       6060     13977   h
   player_id  match_id h_a position
0       8715     14518   h       GK
1       8716     14518   h       DR
2       8816     14518   h       DC
3       6273     14518   h       DC
4       8722     14518   h       DL
